# 04 Orthogonality Threshold

Toy model for how **constraint geometry** affects identifiability.

Core idea:

- higher precision helps
- but precision alone is not enough if constraints are too aligned
- increasing **orthogonality** between constraints can collapse degeneracy

> identifiability depends on both precision and orthogonality


In [ ]:
# Colab setup
!pip -q install numpy matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# grid
x = np.linspace(-2.5, 2.5, 700)
y = np.linspace(-2.5, 2.5, 700)
X, Y = np.meshgrid(x, y)

# base branch structure: two candidate points on a circle
circle = np.abs(X**2 + Y**2 - 2.0) < 0.05

# first constraint: near the diagonal y = x
c1 = np.abs(Y - X) < 0.12

def rotated_band(theta_deg, width=0.12, x0=1.0, y0=1.0):
    theta = np.deg2rad(theta_deg)
    # signed distance to a line through (x0,y0) with direction theta
    # normal vector = (-sin theta, cos theta)
    return np.abs(-(X - x0)*np.sin(theta) + (Y - y0)*np.cos(theta)) < width


## Aligned vs partially rotated vs orthogonal constraints

We keep precision fixed and change only the **angle** of the second constraint.


In [ ]:
angles = [45, 60, 90]
titles = [
    'aligned constraints → degeneracy remains',
    'partially rotated → weaker ambiguity',
    'orthogonal constraints → unique branch'
]

plt.figure(figsize=(12,4))

for i, ang in enumerate(angles):
    plt.subplot(1,3,i+1)
    c2 = rotated_band(ang, width=0.12, x0=1.0, y0=1.0)
    mask = circle & c1 & c2

    plt.contourf(X, Y, mask.astype(float), levels=[-0.5, 0.5, 1.5])

    # reference candidate branches
    if i < 2:
        plt.scatter([-1], [-1], s=55, color='tab:blue', label='wrong')
        plt.scatter([1], [1], s=55, color='tab:orange', label='true')
    else:
        plt.scatter([1], [1], s=55, color='tab:orange', label='true')

    plt.title(titles[i])
    plt.axis('equal')

    if i == 0:
        plt.legend(loc='upper right', frameon=True)

plt.suptitle('Orthogonality threshold → constraint geometry resolves degeneracy')
plt.show()


## Interpretation

- **Aligned constraints** mostly repeat the same information
- **Partially rotated constraints** help, but may still leave ambiguity
- **Orthogonal constraints** contribute genuinely new information and can select a unique solution

Key point:

> precision without orthogonality can fail to identify the true branch
